In [ ]:
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'scripts').is_dir() else Path.cwd().parent
subprocess.run([sys.executable, str(PROJECT_ROOT / 'scripts/prepare_merged_dataset.py')], cwd=PROJECT_ROOT, check=True)

import tensorflow as tf
from tensorflow.keras import layers, models

IMG_SIZE = 224
BATCH_SIZE = 32

# กำหนดคลาสที่ต้องการใช้งาน (ข้ามสารกาแฟดิบ หรือ Green)
TARGET_CLASSES = ['Light', 'Medium', 'Dark']
NUM_CLASSES = len(TARGET_CLASSES)

# โหลดชุดข้อมูลจากโฟลเดอร์ train โดยตรง
train_dataset = tf.keras.utils.image_dataset_from_directory(
    str(PROJECT_ROOT / 'datasets/merged/train'), # เปลี่ยน Path ให้ตรงกับโฟลเดอร์จริง
    labels='inferred',
    class_names=TARGET_CLASSES,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset='training',
    seed=2026
)

# แยก validation จาก train; เก็บ test สำหรับประเมินครั้งสุดท้าย
val_dataset = tf.keras.utils.image_dataset_from_directory(
    str(PROJECT_ROOT / 'datasets/merged/train'), # เปลี่ยน Path ให้ตรงกับโฟลเดอร์จริง
    labels='inferred',
    class_names=TARGET_CLASSES,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset='validation',
    seed=2026
)

print(f"คลาสที่โหลดมาใช้งาน: {train_dataset.class_names}")

# Data Augmentation: สุ่มหมุน พลิก และปรับแสง
data_augmentation = tf.keras.Sequential([
  layers.RandomFlip("horizontal_and_vertical"),
  layers.RandomRotation(0.2),
  layers.RandomZoom(0.15), # สุ่มซูมเข้า/ออก 15%
])

In [ ]:
# 1. โหลด MobileNetV2 แบบดึงค่าน้ำหนักจาก ImageNet (ฉลาดที่สุด)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False # ล็อกค่าน้ำหนักไว้ก่อน

# 2. ประกอบโครงสร้าง
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)

# แปลงค่าพิกเซลจาก 0-255 ให้อยู่ในช่วง -1 ถึง 1 (ตามที่ MobileNetV2 ต้องการเป๊ะๆ)
x = layers.Rescaling(1./127.5, offset=-1)(x)

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

# คอมไพล์ด้วย Learning rate มาตรฐาน
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

In [ ]:
# หยุดเทรนอัตโนมัติถ้าค่า val_loss ไม่ลดลง 3 รอบติดกัน และดึงค่าน้ำหนักรอบที่ดีที่สุดกลับมา
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=3, 
    restore_best_weights=True
)

print("เริ่มการฝึกแบบจำลอง...")
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=50, # เพิ่มรอบสูงสุดเป็น 50
    callbacks=[early_stop]
)

model.save(str(PROJECT_ROOT / 'src/coffee_roast_model.keras'))
print("เทรนเสร็จสิ้น และบันทึกโมเดล .keras สำเร็จ")

In [ ]:
print("กำลังแปลงไฟล์เป็น TensorFlow Lite สำหรับ Raspberry Pi...")

# ดึงโมเดลที่ฝึกเสร็จแล้วมาทำการแปลง
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# สั่งบีบอัดโมเดล (Post-training Quantization) 
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

# บันทึกไฟล์ที่ถูกบีบอัดแล้ว
with open(str(PROJECT_ROOT / 'src/coffee_roast_model_quantized.tflite'), 'wb') as f:
    f.write(tflite_model)
    
print("แปลงไฟล์ .tflite สำเร็จ! นำไฟล์นี้ไปรันบนบอร์ด Pi ได้เลย")

## อัปเดตโมเดลที่แอปใช้
Notebook นี้บันทึกโมเดลเริ่มต้นเท่านั้น หลังเทรนเสร็จให้รันจากโฟลเดอร์โปรเจกต์:
```powershell
python scripts/retrain_pipeline.py
```
ต้องมีภาพ datasets/train, datasets/test และ datasets/real-groups แยก train/val/test ก่อนรัน เพิ่มภาพจริงหลายแหล่ง โดยเฉพาะ Light อย่าใช้ภาพทดสอบเป็นภาพฝึก
Pipeline จะเทรนโมเดลที่แอปโหลด เลือกน้ำหนักด้วย validation และเขียน src/ensemble_config.json จากนั้นรีสตาร์ตแอป
